In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

poly_homopolymer_regions = [309,310,311,16179,16180,16181,16182,16183, 3107]

In [ ]:
df_pacbio = pd.read_csv('../benchmark/pacbio_byTissue/output/ST004-1Q-pacbio/variants/baldur/ST004-1Q-pacbio.mt.baldur.norm.snv_indel.csv',sep=',')
print(len(df_pacbio))
df_ont = pd.read_csv('../benchmark/ont_byTissue/output/ST004-1Q-ont/variants/baldur/ST004-1Q-ont.mt.baldur.norm.snv_indel.csv',sep=',')
print(len(df_ont))


In [ ]:

counts_pacbio = (
    df_pacbio.drop(columns="sample_id")
      .apply(pd.Series.value_counts)
      .fillna(0)
      .T
      .astype(int)
).reset_index()

counts_pacbio = counts_pacbio.rename(columns={
    'index': 'pos',
    0: "REF",
    1: "ALT",
    2: "OTHER",
    -1: "MISSING"
})

counts_pacbio['VAF'] = counts_pacbio['ALT'] / (counts_pacbio['REF'] + counts_pacbio['ALT'] + counts_pacbio['OTHER'])
counts_pacbio['POS'] = counts_pacbio['pos'].str.split(':',expand=True)[0].astype(int)

lf_pacbio = counts_pacbio[counts_pacbio['VAF'] < 0.9][~counts_pacbio['POS'].isin(poly_homopolymer_regions)]
lf_pacbio.sort_values('VAF')

In [ ]:
counts_ont = (
    df_ont.drop(columns="sample_id")
      .apply(pd.Series.value_counts)
      .fillna(0)
      .T
      .astype(int)
).reset_index()


counts_ont = counts_ont.rename(columns={
    'index': 'pos',
    0: "REF",
    1: "ALT",
    2: "OTHER",
    -1: "MISSING"
})

counts_ont['VAF'] = counts_ont['ALT'] / (counts_ont['REF'] + counts_ont['ALT'] + counts_ont['OTHER'])
counts_ont['POS'] = counts_ont['pos'].str.split(':',expand=True)[0].astype(int)

lf_ont = counts_ont[counts_ont['VAF'] < 0.9][~counts_ont['POS'].isin(poly_homopolymer_regions)]
lf_ont.sort_values('VAF')

In [ ]:
common_vars = pd.merge(lf_pacbio,lf_ont, on='pos')['pos'].to_list()
common_vars

In [ ]:
filt_df_pacbio = df_pacbio[['sample_id'] + common_vars]
print(len(filt_df_pacbio))

filt_df_ont = df_ont[['sample_id'] + common_vars]
print(len(filt_df_ont))

In [ ]:
comb_filt_df = pd.concat([filt_df_pacbio, filt_df_ont])
comb_filt_df

In [ ]:
haplotypes = (
    comb_filt_df.drop(columns='sample_id').replace(-1, pd.NA)
      .value_counts(dropna=True)
      .reset_index(name="reads")
)
haplotypes['count_of_ones'] = haplotypes.drop(columns='reads').eq(1).sum(axis=1)
haplotypes

In [ ]:
haplotypes = haplotypes[haplotypes['count_of_ones'] > 0]
haplotypes['freq'] = haplotypes['reads'] / haplotypes['reads'].sum() *100
haplotypes['reads'].sum()

In [ ]:
haplotypes.groupby('count_of_ones')['reads'].sum()

In [ ]:
from matplotlib.colors import ListedColormap

# Filter once so heatmap and frequencies use exactly the same haplotypes
df = haplotypes[haplotypes['reads'] > 0]

fig, (ax, ax_freq) = plt.subplots(
    1, 2,
    figsize=(12, 10),
    gridspec_kw={'width_ratios': [4, 1], 'wspace': 0.05},
    sharey=True
)

my_colors = ["#C2D4C2", "#4F46E5", "#C2D4C2",]
my_cmap = ListedColormap(my_colors)

# Heatmap
sns.heatmap(
    df.drop(columns=['reads', 'count_of_ones', 'freq']),
    annot=False,
    linewidths=0.5,
    linecolor="white",
    cbar=False,
    cmap=my_cmap,
    ax=ax
)

ax.set_xlabel("Variant")
ax.set_ylabel("Haplotype")

y = np.arange(len(df)) + 0.5

# Frequency bar plot
ax_freq.barh(
    y,
    df['reads'].values,
    height=0.8,
    color='steelblue'
)

# # Add freq labels
# for yi, reads, freq in zip(y, df['reads'], df['freq']):
#     ax_freq.text(
#         freq * 1.05,   # slightly to the right of the bar
#         yi,
#         f'{reads}',  # format freq
#         va='center',
#         ha='left'
#     )


ax_freq.set_xlabel("Frequency")
ax_freq.set_ylabel("")
#ax_freq.set_xlim(0,50)
ax_freq.tick_params(axis='y', left=False, labelleft=False)
ax_freq.set_xscale('log')
#ax_freq.invert_yaxis()

plt.tight_layout()
plt.savefig("plots_revisions/suppl/cooccurance.pdf", dpi=300)
plt.show()

In [ ]:
del_df_pacbio = pd.read_csv('../benchmark/pacbio_byTissue/output/ST004-1Q-pacbio/variants/baldur/ST004-1Q-pacbio.mt.baldur.norm.deletions.csv',header=None,names=['sample_id', 'variants'],sep='\t')
print(len(del_df_pacbio))
del_df_ont = pd.read_csv('../benchmark/ont_byTissue/output/ST004-1Q-ont/variants/baldur/ST004-1Q-ont.mt.baldur.norm.deletions.csv',header=None,names=['sample_id', 'variants'],sep='\t')
print(len(del_df_ont))

comb_del_df = pd.concat([del_df_pacbio, del_df_ont])
comb_del_df


In [ ]:
filt_df_w_del = pd.merge(comb_filt_df, comb_del_df, on='sample_id')
filt_df_w_del

In [ ]:
haplotypes_del = (
    filt_df_w_del.drop(columns=['sample_id']).replace(-1, pd.NA)
      .value_counts(dropna=True)
      .reset_index(name="reads")
)

haplotypes_del

In [ ]:
haplotypes_del['count_of_ones'] = haplotypes_del.drop(columns=['reads', 'variants']).eq(1).sum(axis=1)
haplotypes_del

In [ ]:
haplotypes_del.groupby('count_of_ones')['reads'].sum()

In [ ]:
t = haplotypes_del[haplotypes_del['count_of_ones'] > 0].sort_values('reads')
t

t.drop(columns=['variants', 'reads', 'count_of_ones']).value_counts().reset_index()